# Notes

## Introduction to RAG
- RAG gives access to data on which models are not trained
- Answer questions: Collect information --- RETRIEVAL + Reason & respond --- GENERATION
    - Option1: No need to collect information --> Respond based on your knowledge
    - Option2: Extensive research --> synthesize research
- Prompt ->     LLM -> Great answer || Not so great answer -> 1. Very recent information 2. Specialized knowledge
                 |
            Internet data
- What LLMs don't know
    - Private databases
    - Hard to access information
    - Real time data

- **Retriever**
    - Manages knowledge base of trusted information
    - Finds the most relevant information and shares information with the LLM
    - Improves generation

- Application of RAGs : Company chatbots, code author, specialized knowledge, search engine as retrievers

## RAG Architecture Deep Dive
- Normal LLM : Prompt -> LLM -> Response
                Knowledge Base
                    ||
- RAG : Prompt -> Retriever > Relevant documents -> Augmented Prompt -> LLM -> Response

- Advantages of RAG
    - Injects missing knowledge
    - Reduces hallucinations
    - Keeps models upto date
    - Enables source citation
    - Focuses model on generation

## Token
- A piece of a word
- Some words get single tokens, compound words get multiple tokens
- ~10000-100000 tokens in LLMs vocabulary, allowing models to represent any possible word with fewer tokens
- Process current state ---> Calculate probabilities ---> Select next token
- LLM generates a probability distribution of the next token and then randomly chooses the next token from that probability distribution
- **Auto-regressive**
    - Self influencing
    - New tokens make sense in context of the old ones
    - Running the same prompt leads to different completion
## How LLMs learn

```
LLM
 │
 ▼
+--------------------------------------------------+
|                     TRAINING                     |
|                                                  |
|   Neural Network (billions of parameters)        |
|                                                  |
|   Input text with missing tokens                 |
|                                                  |
+--------------------------------------------------+
                       │
                       ▼
                 Predictions
                       │
                       ▼
                   Accurate?
                   /       \
                 No         Yes
                 │           │
                 ▼           ▼
        Update parameters   Continue
                 │
                 └───────────────→ repeat
```
### Why not train LLM on everything
- Higher computational cost
- Context window limit

### Hallucinations
- LLMs generate probable word sequences
- Knowledge gaps cause inaccurate responses
- Truthful != probable

## Retriever
- Share relevant information with LLM which was not available when the model was trained
- Prompt ---> Retriever ---> Knowledge base
- Retriever ranks the documents in the knowledge base basis the relevance to the prompt. Docs with the highest scores are returned
### Retriever tradeoffs
    - Relevance vs irrelevance:- Need to return relevant documents and withhold irrelevant ones
    - Return every document:- Wastes context window
    - Return the single highest ranked document:- Miss valuable information
    - No perfect solution: - Retriever usually doesn't perfectly rank documents
    - Monitor and experiment - Change settings to find what works

### Retriever Architecture
```
+--------+
        | Prompt |
        +--------+
             │
             ▼
      +------------+          +----------------+
      | Retriever  |◄────────►| Knowledge Base |
      +------------+          +----------------+
             │
             ▼
   +--------------------+
   | Relevant Documents |
   +--------------------+
             │
             ▼
   +------------------+
   | Augmented Prompt |
   +------------------+
             │
             ▼
          +-----+
          | LLM |
          +-----+
             │
             ▼
       +----------+
       | Response |
       +----------+
```

#### Two Search Approaches
##### Keyword search
- Looks for documents containing the exact words found in the prompt
- Ensures sensitivity to exact words the user included in the prompt
###### Sparse Vectors
A sparse vector is generated for each document aka __Inverted Index__
- **Frequency based scoring** ---> **Normalized TF Scoring** ---> TF-IDF __(Inverse Document Frequency)__ ---> BM25 __(Best Matching 25)__
- **BM25**
    - Standard keyword search algorith in production retrievers
    - Better performance
    - Same cost
    - More flexibility
##### Semantic search
- Looks for documents with the similar meaning to the prompt
- Finds documents with similar meaning, even without matching words
###### How does it work?
- Prompt and documents each get a vector
- Vectors are compared to generate scores
- The main difference between semantic search and keyword search is how the vectors are assigned
    - **Keyword search**: count words
    - **Semantic search**: use embedding model
        -  Embedding models map token to a location in space, this location is represented by a **vector**
        - Measure vector distance (cosine similarity, dot product)
        - **Higher values, closer vectors**
        - Rank by distance and the return the closest documents

##### Metadata Filtering
- Narrows documents using rigid criteria
- It doesn't perform retrieval, but narrows down the results from other techniques
- Filters based on user attributes, not query content

#### Hybrid search
- Each search returns 20-50 documents. The ranking might differ from the two searches
- Each list filtered on document metadata to ensure only the relevant documents are filtered. **Metadata Filtering** exludes documents based on rigid criteria
- There are 2 filtered lists now, the lists are combined and the top ranked document is returned
- The document is added to the augmented prompt



## Real world applications


# Python refresher

In [14]:
import httpx

# Lists
l1 = ['RAG', 'is', 'awesome']
print(f"Original list: {l1}")

l1.append('!')
print(f"List after adding: {l1}")

l1.remove('awesome')
print(f"List after removing: {l1}")

result = l1.append("this is a test")
print(result)

squares = [x**2 for x in range(1,10)]
print(f"Sqaures: {squares}")

even_squares = [x**2 for x in range (1,10) if x%2==0]
print(even_squares)

person = {
    'name': 'Alice',
    'age': 22,
    'city': 'New York'
}
print(f"Person: {person}")
person['email'] = 'alice@wonderland.com'
print(f"Updated person: {person}")

for key, value in person.items():
    print(f"Key: {key}, value: {value}")

for val in person:
    print(val)

name = "John"
age = 30
greeting = f"Hello, {name}. You are {age} years old."
print(greeting)

people = [
    {
        "name": "Alice Johnson",
        "age": 28,
        "email": "alice.johnson@example.com",
        "location": "New York, NY"
    },
    {
        "name": "Michael Smith",
        "age": 34,
        "email": "michael.smith@example.com",
        "location": "Los Angeles, CA"
    },
    {
        "name": "Emily Davis",
        "age": 22,
        "email": "emily.davis@example.com",
        "location": "Austin, TX"
    },
    {
        "name": "John Brown",
        "age": 45,
        "email": "john.brown@example.com",
        "location": "Chicago, IL"
    },
    {
        "name": "Sarah Wilson",
        "age": 31,
        "email": "sarah.wilson@example.com",
        "location": "Seattle, WA"
    }
]
# Expected output
# ```
# Name: Alice Johnson, Age: 28, E-mail: alice.johnson@example.com, Location: New York, NY
# Name: Michael Smith, Age: 34, E-mail: michael.smith@example.com, Location: Los Angeles, CA
# ... (continues)
# ```

print("---------------------------------------------------------")
for person in people:
    output = f"Name: {person['name']}, Age: {person['age']}, Email: {person['email']}, Location: {person['location']}"
    print(output)
print("---------------------------------------------------------")

Original list: ['RAG', 'is', 'awesome']
List after adding: ['RAG', 'is', 'awesome', '!']
List after removing: ['RAG', 'is', '!']
None
Sqaures: [1, 4, 9, 16, 25, 36, 49, 64, 81]
[4, 16, 36, 64]
Person: {'name': 'Alice', 'age': 22, 'city': 'New York'}
Updated person: {'name': 'Alice', 'age': 22, 'city': 'New York', 'email': 'alice@wonderland.com'}
Key: name, value: Alice
Key: age, value: 22
Key: city, value: New York
Key: email, value: alice@wonderland.com
name
age
city
email
Hello, John. You are 30 years old.
---------------------------------------------------------
Name: Alice Johnson, Age: 28, Email: alice.johnson@example.com, Location: New York, NY
Name: Michael Smith, Age: 34, Email: michael.smith@example.com, Location: Los Angeles, CA
Name: Emily Davis, Age: 22, Email: emily.davis@example.com, Location: Austin, TX
Name: John Brown, Age: 45, Email: john.brown@example.com, Location: Chicago, IL
Name: Sarah Wilson, Age: 31, Email: sarah.wilson@example.com, Location: Seattle, WA
------

# LLM Calls & Crafting Simple Augmented Prompts

## Generate with single input
This function allows to generate text from a language model based on a single input prompt

In [21]:
from utils.utils_openAI import generate_with_single_input
from dotenv import load_dotenv

load_dotenv()

output = generate_with_single_input(prompt = "What is the capital of France?")
print ("Role: ", output['role'])
print ("Content: ", output['content'])

Role:  assistant
Content:  The capital of France is Paris.


## Generate with multiple input
This function is designed to handle multiple input messages in a conversational context. The input format is a dictionary with two keys:

1. 'role' - the role that the message is being passed (usually assistant, system or user)
2. 'content' - the prompt

In [22]:
from utils.utils_openAI import generate_with_multiple_input
from dotenv import load_dotenv

load_dotenv()
messages = [
    {'role': 'user', 'content': 'Hello, who won the FIFA world cup in 2018?'},
    {'role': 'assistant', 'content': 'France won the 2018 FIFA World Cup.'},
    {'role': 'user', 'content': 'Who was the captain?'}
]
output = generate_with_multiple_input(
    messages=messages,
    max_tokens=100
)
print("Role: ", output['role'])
print("Content: ", output['content'])

Role:  assistant
Content:  The captain of the French team that won the 2018 FIFA World Cup was Hugo Lloris.


## Integration of data into an LLM prompt

### Data

In [23]:
house_data = [
    {
        "address": "123 Maple Street",
        "city": "Springfield",
        "state": "IL",
        "zip": "62701",
        "bedrooms": 3,
        "bathrooms": 2,
        "square_feet": 1500,
        "price": 230000,
        "year_built": 1998
    },
    {
        "address": "456 Elm Avenue",
        "city": "Shelbyville",
        "state": "TN",
        "zip": "37160",
        "bedrooms": 4,
        "bathrooms": 3,
        "square_feet": 2500,
        "price": 320000,
        "year_built": 2005
    }
]

### Create the prompt

In [24]:
def house_info_layout(houses):
    layout = ''
    for house in houses:
        layout += (f"House located at {house['address']}, {house['city']}, {house['state']} {house['zip']} with "
                   f"{house['bedrooms']} bedrooms, {house['bathrooms']} bathrooms, "
                   f"{house['square_feet']} sq feet area, priced at ${house['price']}, "
                   f"built in {house['year_built']}.\n"
                   )
    return layout

In [25]:
print(house_info_layout(house_data))

House located at 123 Maple Street, Springfield, IL 62701 with 3 bedrooms, 2 bathrooms, 1500 sq feet area, priced at $230000, built in 1998.
House located at 456 Elm Avenue, Shelbyville, TN 37160 with 4 bedrooms, 3 bathrooms, 2500 sq feet area, priced at $320000, built in 2005.



In [26]:
def generate_prompt(query, houses):
    houses_layout = house_info_layout(houses)
    prompt = f"""
User the following houses information to answer user queries.
{houses_layout}
Query: {query}
"""
    return prompt

In [27]:
print(generate_prompt("Which is the most expensive house", house_data))


User the following houses information to answer user queries.
House located at 123 Maple Street, Springfield, IL 62701 with 3 bedrooms, 2 bathrooms, 1500 sq feet area, priced at $230000, built in 1998.
House located at 456 Elm Avenue, Shelbyville, TN 37160 with 4 bedrooms, 3 bathrooms, 2500 sq feet area, priced at $320000, built in 2005.

Query: Which is the most expensive house



In [29]:
query = "Which is the most expensive house? And the biggest one?"
query_without_house_info = generate_with_single_input(prompt = query, role='user')
print(query_without_house_info)
enhanced_query = generate_prompt(query, house_data)
query_with_house_info = generate_with_single_input(prompt = enhanced_query, role='assistant')

print(query_without_house_info['content'])
print("------------------------------------------------")
print(query_with_house_info['content'])

{'role': 'assistant', 'content': 'Could you please clarify which location or context you are referring to when you ask about the most expensive and biggest house? For example, are you asking about the most expensive and biggest house worldwide, in a specific country or city, or something else? This will help me provide the most accurate and relevant information.'}
Could you please clarify which location or context you are referring to when you ask about the most expensive and biggest house? For example, are you asking about the most expensive and biggest house worldwide, in a specific country or city, or something else? This will help me provide the most accurate and relevant information.
------------------------------------------------
The most expensive house is the one located at 456 Elm Avenue, Shelbyville, TN 37160, priced at $320,000. It is also the biggest house, with an area of 2,500 square feet.
